# 01 · Explore the historical MBTA bus arrival/departure data
One monthly file, opened before anything is loaded into Postgres. Six questions from Task 2;
write your answer under each one.

In [ ]:
import glob
import pandas as pd

path = sorted(glob.glob("../data/raw/*.csv"))[0]
df = pd.read_csv(path, dtype=str)
print(path, f"{len(df):,} rows")
df.head()

## 1. How are the times encoded?

In [ ]:
print(df["scheduled"].head(3).tolist())
print("date part of scheduled:", df["scheduled"].str[:10].value_counts().to_dict())

**Answer:** 

## 2. How many rows have no `actual`?

In [ ]:
print(f"{df['actual'].isna().sum():,} rows ({df['actual'].isna().mean():.1%})")

**Answer:** 

## 3. What does the delay look like?

In [ ]:
scheduled = pd.to_datetime(df["scheduled"].str[:19])
actual = pd.to_datetime(df["actual"].str[:19])
delay = (actual - scheduled).dt.total_seconds() / 60
print(delay.describe(percentiles=[.01, .05, .5, .95, .99]).round(2))
print("outside [-30, +120]:", ((delay < -30) | (delay > 120)).sum())
print("late rate (> 3 min):", round((delay[(delay >= -30) & (delay <= 120)] > 3).mean(), 4))


**Answer:** 

## 4. Is the key (service date, route, direction, stop, scheduled) unique?

In [ ]:
key = ["service_date", "route_id", "direction_id", "stop_id", "scheduled"]
dup = df.duplicated(key, keep=False)
print(f"{dup.sum():,} rows share a key with another row")
df[dup].sort_values(key).head(6)

**Answer:** 

## 5. Which columns are half empty, and why?

In [ ]:
print(df.isna().mean().round(3).sort_values(ascending=False))
pd.crosstab(df["standard_type"], df["scheduled_headway"].isna())

**Answer:** 

## 6. What do the route and stop ids look like?

In [ ]:
routes = sorted(df["route_id"].unique())
print(len(routes), "routes:", routes[:12], "...", routes[-8:])
print("stop ids:", df["stop_id"].drop_duplicates().head(10).tolist())

**Answer:** 